# MiniLM Baseline Retrieval — Summary-Only Variant
## Dense semantic search using all-MiniLM-L6-v2 (384-d embeddings)

**What this notebook does:**

Implements the MiniLM dense retrieval baseline using *only* the free-text `summary`
field — no structured fields (name, country, industry code, keywords). This is
the "Set A" baseline: how well does MiniLM do with just the description text
alone, as a comparison point against `03_baseline_minilm.ipynb`'s all-fields
variant.

**How it works (plain English):**
Every company description is converted into a list of 384 numbers (an *embedding*)
that captures its meaning. At search time, the query is also converted into 384
numbers, and we find companies whose numbers are most similar — even if they use
completely different words from the query.

**MiniLM vs BGE-large:**
| | MiniLM | BGE-large |
|---|---|---|
| Parameters | 22M | 335M |
| Embedding dims | 384 | 1024 |
| GPU needed | No (fast on CPU) | Yes (slow on CPU) |
| FAISS index | `IndexFlatL2` | `IndexFlatIP` |
| Normalisation | Not required | Required |

> **Why different FAISS index types?**
> MiniLM uses L2 (Euclidean) distance — `IndexFlatL2`.
> BGE uses cosine similarity (inner product on normalised vectors) — `IndexFlatIP`.
> Using the wrong index for a model gives incorrect rankings.

**What is being embedded:**
Only the `summary` field is embedded for this variant — no name, country, state,
city, industry (NACE), size, or keywords. This isolates how much retrieval
signal comes from the description text alone.

**Folder structure:**
```
result/
└── 03_baseline_minilm_summary/
    ├── company_embeddings.npy    # Pre-computed MiniLM embeddings (98716 x 384)
    ├── company_faiss.index       # FAISS IndexFlatL2 index
    ├── minilm_results.csv        # Top-1000 results per query (101 queries)
    └── evaluation_minilm.csv     # NDCG, Precision, Recall, F1 @ k in {10,50,100,1000}
```

### Notebook structure
1. Environment setup
2. Imports
3. Load dataset & build corpus (summary field only)
4. GPU check
5. Encode all companies with MiniLM
6. Build FAISS index
7. Run all 101 queries
8. Evaluation — NDCG, Precision, Recall, F1 @ k in {10, 50, 100, 1000}
9. Final summary

## 1 · Environment Setup

Load environment variables and create the output folder.
The folder is created automatically if it does not exist.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
API_KEY  = os.getenv('API_KEY')
BASE_URL = os.getenv('BASE_URL')

RESULT_DIR = Path('result/03_baseline_minilm_summary')
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print(f'[Setup] Result folder : {RESULT_DIR}/ — ready')

## 2 · Imports

| Package | Role |
|---|---|
| `sentence_transformers` | Loads `all-MiniLM-L6-v2` — produces 384-d embeddings |
| `faiss` | Facebook AI Similarity Search — fast vector nearest-neighbour lookup |
| `torch` | PyTorch — used to check if GPU is available |
| `pandas / numpy` | Data loading and numerical operations |
| `time` | Measuring encoding and query latency |

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import torch
import json, time
import numpy as np
import pandas as pd
from pathlib import Path
import nltk
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

RESULT_DIR = Path('result/03_baseline_minilm_summary')
print('[Imports] All packages loaded successfully')

## 3 · Load Dataset & Build Corpus

**Source:** `dataset/production_results.xlsx`

We de-duplicate on `domain` to get one row per company (98,716 unique companies).

**Why summary-only for this variant?**
This is the "Set A" baseline: embed *only* the free-text `summary` field, with
no structured fields (name, country, industry code, keywords) to fall back on.
This isolates how much retrieval quality MiniLM gets from the summary text alone,
as a comparison point against `03_baseline_minilm.ipynb`'s all-fields variant.

In [ ]:
print('[Load] Loading production results...')
results_df = pd.read_excel('dataset/production_results.xlsx')
print(f'[Load] Total rows        : {len(results_df):,}')
print(f'[Load] Columns available : {list(results_df.columns)}')

# De-duplicate: one row per company 
all_companies = results_df.drop_duplicates(subset='domain').reset_index(drop=True)
print(f'[Load] Unique companies  : {len(all_companies):,}')

# Load queries
with open('dataset/goi_search_results.json', 'r') as f:
    data = json.load(f)
print(f'[Load] Queries           : {len(data)}')

# Build rich text per company combining ALL available fields 
# Same function as BGE notebook — identical input for fair comparison
print('[Load] Building rich text for each company...')

def build_rich_text(row):
    """Summary only — Set A baseline experiments."""
    return str(row.get('summary', '')) if pd.notna(row.get('summary')) else ''

rich_texts = [build_rich_text(row) for _, row in all_companies.iterrows()]

print(f'[Load] Sample rich text (first company):')
print(f'  {rich_texts[0][:300]}...')

## 4 · GPU Check

MiniLM is a small model (22M parameters) and **can run on CPU** at reasonable speed.
However GPU will be faster — especially for encoding 99k companies.

**Important difference from BGE:**
MiniLM does NOT require `normalize_embeddings=True` and uses `IndexFlatL2`
(Euclidean distance) — NOT `IndexFlatIP`.
Using `IndexFlatIP` with unnormalised MiniLM embeddings would give wrong results.

In [ ]:
print(f'[GPU] CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'[GPU] Device          : {torch.cuda.get_device_name(0)}')
    print(f'[GPU] VRAM            : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    DEVICE = 'cuda'
else:
    print('[GPU] No GPU — running on CPU (MiniLM is fast enough on CPU)')
    DEVICE = 'cpu'
print(f'[GPU] Using device    : {DEVICE}')

## 5 · Encode All Companies with MiniLM

**Model:** `all-MiniLM-L6-v2`
- 6-layer distilled MiniLM, fine-tuned on sentence-pair similarity tasks
- Produces 384-dimensional embeddings (vs 1024-d for BGE)
- Does NOT require normalisation — use raw embeddings with `IndexFlatL2`

**Encoding time:**
- CPU: ~2-3 minutes at batch size 64
- GPU: ~1 minute at batch size 256

Embeddings are saved immediately to avoid re-running if the kernel restarts.

In [ ]:
print('[Encode] Loading MiniLM model...')
t0    = time.time()
model = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)
print(f'[Encode] Model loaded in  : {time.time()-t0:.1f}s  on {model.device}')
print(f'[Encode] Embedding dims   : 384')
print(f'[Encode] Parameters       : ~22M')

print('[Encode] Encoding all companies...')
# MiniLM does NOT need normalize_embeddings — uses L2 distance
batch_size = 256 if DEVICE == 'cuda' else 64
print(f'[Encode] Batch size       : {batch_size}')
t0 = time.time()

embeddings = model.encode(
    rich_texts,
    batch_size=batch_size,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=False,  # NOT required for MiniLM — uses L2 distance
)

ENCODE_TIME = time.time() - t0
print(f'[Encode] Done in          : {ENCODE_TIME/60:.1f} minutes')
print(f'[Encode] Embeddings shape : {embeddings.shape}')  # (98716, 384)

np.save(RESULT_DIR / 'company_embeddings.npy', embeddings)
print(f'[Encode] Saved to         : {RESULT_DIR}/company_embeddings.npy')

## 6 · Build FAISS Index

**Index type: `IndexFlatL2`** (Flat L2 / Euclidean distance)

MiniLM embeddings are NOT normalised, so we use Euclidean distance.
Lower L2 distance = more similar company.

> **Contrast with BGE:** BGE uses `IndexFlatIP` (inner product = cosine similarity
> for normalised vectors). MiniLM uses `IndexFlatL2` (Euclidean distance).
> Never mix these up — using the wrong index for a model gives incorrect rankings.

In [ ]:
print('[FAISS] Loading embeddings...')
embeddings = np.load(RESULT_DIR / 'company_embeddings.npy').astype('float32')
print(f'[FAISS] Embeddings shape  : {embeddings.shape}')

print('[FAISS] Building IndexFlatL2...')
t0        = time.time()
dimension = embeddings.shape[1]  # 384
index     = faiss.IndexFlatL2(dimension)  # L2 distance for non-normalised MiniLM
index.add(embeddings)
INDEX_BUILD_TIME = time.time() - t0
print(f'[FAISS] Index built in    : {INDEX_BUILD_TIME:.2f}s')
print(f'[FAISS] Vectors in index  : {index.ntotal:,}')
print(f'[FAISS] Index type        : IndexFlatL2 (exact L2 distance)')

faiss.write_index(index, str(RESULT_DIR / 'company_faiss.index'))
print(f'[FAISS] Index saved to    : {RESULT_DIR}/company_faiss.index')

## 7 · Run MiniLM Across All 101 Queries

For each query:
1. Encode the query string with MiniLM (no normalisation)
2. Search the FAISS L2 index for top-1,000 nearest companies
3. Store results with rank and L2 distance score

**Note on scores:** FAISS `IndexFlatL2` returns **distances** (lower = better),
not similarities. We store them as-is — lower score means more relevant.

**Output:** `result/03_baseline_minilm_summary/minilm_results.csv`

In [ ]:
print(f'[Run] Starting MiniLM retrieval for {len(data)} queries...')
print(f'[Run] Retrieving top-1000 per query')
print('-' * 55)

all_minilm_results = []
query_times        = []
total_start        = time.time()

for i, item in enumerate(data):
    query_id = item['query_id']
    query    = item['query']

    # Encode query + search — time both together
    t0        = time.perf_counter()
    query_emb = model.encode(
        [query],
        normalize_embeddings=False,  # NOT normalised for MiniLM
        convert_to_numpy=True
    ).astype('float32')
    distances, idxs = index.search(query_emb, 1000)
    query_ms  = (time.perf_counter() - t0) * 1000
    query_times.append(query_ms)

    for rank, (idx, dist) in enumerate(zip(idxs[0], distances[0])):
        company = all_companies.iloc[idx]
        all_minilm_results.append({
            'query_id': query_id,
            'query':    query,
            'rank':     rank + 1,
            'score':    float(dist),  # L2 distance — lower = more similar
            'domain':   company['domain'],
            'name':     company.get('name', ''),
            'country':  company.get('country', ''),
            'summary':  company.get('summary', ''),
        })

    if (i + 1) % 20 == 0 or (i + 1) == len(data):
        elapsed   = time.time() - total_start
        remaining = (len(data) - i - 1) * elapsed / (i + 1)
        print(f'[Run] {i+1:3d}/{len(data)}  |  '
              f'avg {sum(query_times)/len(query_times):.1f}ms/query  |  '
              f'~{remaining:.0f}s remaining')

minilm_df = pd.DataFrame(all_minilm_results)
minilm_df.to_csv(RESULT_DIR / 'minilm_results.csv', index=False)

AVG_LATENCY_MS = sum(query_times) / len(query_times)
print('-' * 55)
print(f'[Run] Done!')
print(f'[Run] Total results       : {len(minilm_df):,}')
print(f'[Run] Avg query latency   : {AVG_LATENCY_MS:.1f}ms')
print(f'[Run] Saved to            : {RESULT_DIR}/minilm_results.csv')

## 8 · Evaluation — NDCG, Precision, Recall, F1 @ k

Same evaluation protocol as BM25 and BGE notebooks — results are directly comparable.

### Pseudo-relevance labels
A company is **relevant** for a query if it appears in the **production top-100**.

### Metrics

| Metric | Formula | What it measures |
|---|---|---|
| **Precision@k** | Relevant in top-k / k | Quality of the top-k results |
| **Recall@k** | Relevant in top-k / Total relevant | Coverage — how many relevant companies found |
| **F1@k** | 2 x P x R / (P + R) | Harmonic mean — balances Precision and Recall |
| **NDCG@k** | DCG@k / IDCG@k | Ranking quality — rewards relevant results ranked higher |

All metrics reported at k in {10, 50, 100, 1000}.

In [ ]:
print('[Eval] Loading production labels...')
production_df = pd.read_excel('dataset/production_results.xlsx')

K_VALUES = [10, 50, 100, 500, 1000]

def get_relevant(query_id, top_k=1000):
    return set(production_df[
        (production_df['query_id'] == query_id) &
        (production_df['rank'] <= top_k)
    ]['domain'].tolist())

def precision_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / k if k else 0

def recall_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / len(relevant) if relevant else 0

def f1_at_k(retrieved, relevant, k):
    p = precision_at_k(retrieved, relevant, k)
    r = recall_at_k(retrieved, relevant, k)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0

def dcg_at_k(retrieved, relevant, k):
    return sum(
        1 / np.log2(i + 2)
        for i, d in enumerate(retrieved[:k]) if d in relevant
    )

def ndcg_at_k(retrieved, relevant, k):
    ideal = dcg_at_k(list(relevant), relevant, k)
    return dcg_at_k(retrieved, relevant, k) / ideal if ideal else 0

# Evaluate all queries
print('[Eval] Computing metrics for all queries...')
eval_rows = []

for i, item in enumerate(data):
    qid       = item['query_id']
    query     = item['query']
    relevant  = get_relevant(qid)
    retrieved = (
        minilm_df[minilm_df['query_id'] == qid]
        .sort_values('rank')['domain'].tolist()
    )
    for k in K_VALUES:
        eval_rows.append({
            'query_id':  qid,
            'query':     query,
            'k':         k,
            'precision': precision_at_k(retrieved, relevant, k),
            'recall':    recall_at_k(retrieved, relevant, k),
            'f1':        f1_at_k(retrieved, relevant, k),
            'ndcg':      ndcg_at_k(retrieved, relevant, k),
        })

    if (i + 1) % 25 == 0:
        print(f'[Eval] {i+1}/101 queries evaluated...')

eval_df = pd.DataFrame(eval_rows)
eval_df.to_csv(RESULT_DIR / 'evaluation_minilm.csv', index=False)
print(f'[Eval] Saved to {RESULT_DIR}/evaluation_minilm.csv')

## 9 · Final Summary

Full results averaged across all 101 queries.

**Expected pattern vs BGE:**
MiniLM (22M params, 384-d) should score lower than BGE (335M params, 1024-d)
on all quality metrics. However MiniLM may be faster on CPU since it runs
without GPU. The key finding from previous experiments was that MiniLM is
actually **slower** than BGE at query time because BGE runs on GPU while
MiniLM runs on CPU — showing that hardware matters more than model size for latency.

In [ ]:
print('[Summary] ============================================================')
print('[Summary] MiniLM BASELINE RESULTS')
print(f'\n[Summary] Encoding time     : {ENCODE_TIME/60:.1f} minutes')
print(f'[Summary] Index build time  : {INDEX_BUILD_TIME:.2f}s')
print(f'[Summary] Avg query latency : {AVG_LATENCY_MS:.1f}ms')
print(f'[Summary] Companies encoded : {len(all_companies):,}')
print(f'[Summary] Embedding dims    : 384')
print(f'[Summary] FAISS index type  : IndexFlatL2')
print(f'[Summary] Normalisation     : False (not required for MiniLM)')
print(f'[Summary] Queries evaluated : {len(data)}')
print()
print(f'  {"k":<6} {"NDCG":>8} {"Precision":>10} {"Recall":>8} {"F1":>8}')
print('  ' + '-' * 46)
for k in K_VALUES:
    sub = eval_df[eval_df['k'] == k]
    print(f'  {k:<6} '
          f'{sub["ndcg"].mean():>8.3f} '
          f'{sub["precision"].mean():>10.3f} '
          f'{sub["recall"].mean():>8.3f} '
          f'{sub["f1"].mean():>8.3f}')